# Apache Spark Environment Setup and Verification

This notebook verifies that your local Apache Spark development environment is properly configured and ready for use. Follow the setup guide in `SETUP_GUIDE.md` before running this notebook.

## Prerequisites
- Java 11 installed and JAVA_HOME configured
- Python 3.10 or higher installed
- Apache Spark 3.5.0 downloaded and SPARK_HOME configured
- PySpark and Jupyter installed via pip

## Step 1: Verify System Environment Variables

Before running any Spark code, verify that environment variables are properly set up.

In [ ]:
import os
import sys

print("=" * 60)
print("ENVIRONMENT VERIFICATION")
print("=" * 60)

# Check Python version
print(f"\n✓ Python Version: {sys.version}")
print(f"  Python Executable: {sys.executable}")

# Check environment variables
env_vars = {
    'JAVA_HOME': 'Java Installation Directory',
    'SPARK_HOME': 'Spark Installation Directory',
    'PYSPARK_PYTHON': 'Python for PySpark',
}

print("\nEnvironment Variables:")
for var, description in env_vars.items():
    value = os.environ.get(var, "NOT SET")
    status = "✓" if var in os.environ else "✗"
    print(f"  {status} {var}: {value}")
    if description:
        print(f"     ({description})")

# Check if Java is accessible
print("\nJava Accessibility:")
try:
    import subprocess
    result = subprocess.run(['java', '-version'], capture_output=True, text=True)
    if result.returncode == 0:
        print(f"  ✓ Java is accessible")
        print(f"    {result.stderr.split(chr(10))[0]}")
    else:
        print(f"  ✗ Java not found in PATH")
except Exception as e:
    print(f"  ✗ Error checking Java: {str(e)}")

print("\n" + "=" * 60)

## Step 2: Import PySpark and Create Spark Session

Initialize a Spark session configured for local development. This is the entry point for all Spark functionality.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
import pyspark

print("=" * 60)
print("PYSPARK IMPORT AND SESSION CREATION")
print("=" * 60)

print(f"\n✓ PySpark Version: {pyspark.__version__}")

# Create Spark Session with optimized configuration
spark = SparkSession.builder \
    .appName("Retail Analytics Setup Test") \
    .master("local[*]") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

print(f"\n✓ Spark Session Created: {spark.appName}")
print(f"  Master: {spark.sparkContext.master()}")
print(f"  App ID: {spark.sparkContext.applicationId}")

# Get SparkContext
sc = spark.sparkContext
print(f"\n✓ Spark Context Configuration:")
print(f"  Default Parallelism: {sc.defaultParallelism}")
print(f"  Default Min Partitions: {sc.defaultMinPartitions}")

print("\n" + "=" * 60)

## Step 3: Test RDD Operations (Resilient Distributed Datasets)

RDDs are the fundamental data structure of Spark. Test basic RDD operations to verify distributed computing.

In [ ]:
print("=" * 60)
print("RDD OPERATIONS TEST")
print("=" * 60)

# Test 1: Create RDD from collection
print("\nTest 1: Create and transform RDD")
data = range(1, 51)
rdd = sc.parallelize(data, numPartitions=4)
print(f"  ✓ Created RDD with {rdd.count()} elements")
print(f"    Partitions: {rdd.getNumPartitions()}")

# Test 2: Map operation
squared_rdd = rdd.map(lambda x: x ** 2)
sum_squared = squared_rdd.sum()
print(f"  ✓ Map operation: Sum of squares (1-50)² = {sum_squared:,}")

# Test 3: Filter operation
even_rdd = rdd.filter(lambda x: x % 2 == 0)
even_count = even_rdd.count()
print(f"  ✓ Filter operation: Found {even_count} even numbers")

# Test 4: Reduce operation
product = rdd.take(5)
print(f"  ✓ Take operation: First 5 elements = {product}")

print("\n" + "=" * 60)

## Step 4: Test Spark DataFrame Operations

DataFrames are distributed data structures similar to pandas DataFrames but optimized for Spark's distributed computing.

In [ ]:
print("=" * 60)
print("DATAFRAME OPERATIONS TEST")
print("=" * 60)

# Test 1: Create DataFrame from list
print("\nTest 1: Create DataFrame from list")
data_list = [
    (1, "Alice", 45000),
    (2, "Bob", 55000),
    (3, "Charlie", 65000),
    (4, "David", 50000),
    (5, "Eve", 70000)
]

schema = StructType([
    StructField("employee_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("salary", IntegerType(), True)
])

df = spark.createDataFrame(data_list, schema=schema)
print(f"  ✓ Created DataFrame with {df.count()} rows and {len(df.columns)} columns")
print(f"    Columns: {df.columns}")

# Test 2: Display DataFrame
print("\nTest 2: Display DataFrame")
df.show()

# Test 3: DataFrame transformations
print("\nTest 3: DataFrame transformations")
df_filtered = df.filter(df.salary > 50000)
print(f"  ✓ Filtered rows with salary > 50000: {df_filtered.count()} employees")
df_filtered.show()

# Test 4: DataFrame aggregations
print("\nTest 4: DataFrame aggregations")
from pyspark.sql.functions import avg, sum, count, max, min
agg_result = df.agg(
    avg('salary').alias('avg_salary'),
    max('salary').alias('max_salary'),
    min('salary').alias('min_salary'),
    count('*').alias('total_employees')
)
print("  ✓ Aggregation results:")
agg_result.show()

print("\n" + "=" * 60)

## Step 5: Test Spark SQL

Spark SQL enables you to write SQL queries against DataFrames. This is powerful for data exploration and analysis.